# SmolVLM 256M experiment record
ETA source variant; partial execution is recorded in notebook 07.

Original: `EdgeVLM_256M_Phase2_Finetuning_with_ETA.ipynb`. Numeric/text outputs are retained where present. Embedded visual outputs and image-path previews are removed for publication. Execution counts are not fabricated. See README.md and the source manifest.

This is a historical server notebook, not a portable CLI. Review paths, dependencies, adapter initialization and optional export cells before executing. No dataset or weights are included.


# EdgeVLM Phase 2 — SmolVLM-256M continued fine-tuning

This notebook starts from the last class-weighted SmolVLM-256M LoRA adapter,
continues fine-tuning on the new Phase 2 training split, selects the best
checkpoint using the Phase 2 validation split, and evaluates once on the full
Phase 2 test split.

Important rules:

- Run the cells from top to bottom.
- The four labels are `NORMAL`, `DROWSY`, `LOOKING_AWAY`, and `PHONE_USAGE`.
- `ear_value`, `mediapipe_status`, and the ground-truth label are never given to
  the model as input.
- The old adapter is read only. All new checkpoints and results are written to
  a separate `phase2_smolvlm_256m` folder.


## 1. Locate the Phase 2 data and configure the run


In [ ]:
import os
import sys
from pathlib import Path

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if "transformers" in sys.modules or "tensorflow" in sys.modules:
    raise RuntimeError(
        "Transformers or TensorFlow was already imported in this kernel. "
        "Use Kernel > Restart Kernel, then run this notebook from the first cell."
    )

HOME = Path("/home/jovyan")

split_candidates = [
    HOME / "phase2-tain-val-test-splits",
    HOME / "phase2-train-val-test-splits",
]
SPLIT_DIR = next((p for p in split_candidates if p.is_dir()), split_candidates[0])

TRAIN_CSV = SPLIT_DIR / "final_train.csv"
VAL_CSV = SPLIT_DIR / "final_val.csv"
TEST_CSV = SPLIT_DIR / "final_test.csv"

project_candidates = [
    HOME / "EdgeVLM_finetune vlm256",
    HOME / "EdgeVLM_finetune_vlm256",
    HOME / "EdgeVLM_finetune",
]
PROJECT_ROOT = next((p for p in project_candidates if p.is_dir()), project_candidates[0])
OUTPUT_ROOT = PROJECT_ROOT / "phase2_smolvlm_256m"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
FINAL_ADAPTER_DIR = OUTPUT_ROOT / "final_adapter"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

LABELS = ["NORMAL", "DROWSY", "LOOKING_AWAY", "PHONE_USAGE"]
MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
IMAGE_LONGEST_EDGE = 512
SEED = 42

print("Split directory:", SPLIT_DIR)
print("Project directory:", PROJECT_ROOT)
print("New output directory:", OUTPUT_ROOT)
for path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    print(path, "exists:", path.is_file())
    if not path.is_file():
        raise FileNotFoundError(path)


## 2. Read and validate all three CSV files


In [ ]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

datasets = {"train": train_df, "validation": val_df, "test": test_df}
required_columns = {"frame_path", "label", "subject", "source_dataset"}

for split_name, dataframe in datasets.items():
    missing_columns = required_columns - set(dataframe.columns)
    if missing_columns:
        raise ValueError(f"{split_name} is missing columns: {sorted(missing_columns)}")

    dataframe["label"] = dataframe["label"].astype(str).str.strip().str.upper()
    unknown = sorted(set(dataframe["label"]) - set(LABELS))
    if unknown:
        raise ValueError(f"{split_name} contains unknown labels: {unknown}")

    print(f"\n{split_name}: {len(dataframe)} rows")
    print(dataframe["label"].value_counts().reindex(LABELS, fill_value=0))

print("\nTotal rows:", sum(len(df) for df in datasets.values()))


## 3. Verify image paths, duplicates, and subject separation


In [ ]:
def normalized_path(value):
    return str(value).strip().replace("\\", "/")


def resolve_image_path(value):
    path = Path(normalized_path(value))
    if not path.is_absolute():
        path = HOME / path
    return path


for split_name, dataframe in datasets.items():
    dataframe["resolved_path"] = dataframe["frame_path"].map(
        lambda value: str(resolve_image_path(value))
    )
    missing = [p for p in dataframe["resolved_path"] if not Path(p).is_file()]
    duplicate_count = int(dataframe["resolved_path"].duplicated().sum())
    print(
        split_name,
        "| missing images:", len(missing),
        "| duplicate rows:", duplicate_count,
    )
    if missing:
        print("First missing image:", missing[0])

if any(not Path(p).is_file() for df in datasets.values() for p in df["resolved_path"]):
    raise FileNotFoundError("At least one image is missing. Fix paths before training.")

# Exact duplicate image paths must not cross splits.
for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    overlap = set(datasets[left]["resolved_path"]) & set(datasets[right]["resolved_path"])
    print(f"Exact image overlap {left}/{right}:", len(overlap))
    if overlap:
        raise ValueError(f"Image leakage detected between {left} and {right}")

# Combine dataset name and subject because subject numbers may repeat across datasets.
def subject_keys(dataframe):
    return set(
        zip(
            dataframe["source_dataset"].astype(str),
            dataframe["subject"].astype(str),
        )
    )

for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    overlap = subject_keys(datasets[left]) & subject_keys(datasets[right])
    print(f"Subject overlap {left}/{right}:", len(overlap))
    if overlap:
        print("WARNING: subject leakage examples:", list(overlap)[:5])

# Remove repeated paths inside training in memory only. Validation and test remain unchanged.
before = len(train_df)
train_df = train_df.drop_duplicates(subset=["resolved_path"], keep="first").reset_index(drop=True)
datasets["train"] = train_df
print("Training duplicates removed in memory:", before - len(train_df))
print("Final training rows:", len(train_df))


If any subject-overlap warning appears, stop and review the split before
fine-tuning. A driver appearing in both training and test data can make the
reported result unrealistically high.


## 4. Install the compatible fine-tuning libraries


In [ ]:
%pip install -q \
  "transformers==4.46.3" \
  "tokenizers==0.20.3" \
  "peft==0.20.0" \
  "accelerate>=1.0,<2" \
  "safetensors>=0.4" \
  "sentencepiece" \
  "scikit-learn" \
  "tqdm"


After the installation cell finishes, **restart the kernel even if Jupyter does
not ask you to**, then run the notebook again from Cell 1. This is required so
Transformers reads `USE_TF=0` before anything imports TensorFlow. The project
does not use TensorFlow.


## 5. Verify the runtime and locate the previous good adapter


In [ ]:
import sys
import torch
import transformers
import tokenizers
import peft
import accelerate

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this fine-tuning notebook.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)

adapter_candidates = [
    HOME / "EdgeVLM_finetune vlm256" / "smolvlm_256m_lora_augmented_classweighted",
    HOME / "EdgeVLM_finetune_vlm256" / "smolvlm_256m_lora_augmented_classweighted",
    HOME / "EdgeVLM_finetune" / "smolvlm_256m_lora_augmented_classweighted",
]
PREVIOUS_ADAPTER_DIR = next(
    (p for p in adapter_candidates if (p / "adapter_model.safetensors").is_file()),
    None,
)

if PREVIOUS_ADAPTER_DIR is None:
    raise FileNotFoundError(
        "The previous class-weighted adapter was not found. Checked:\n"
        + "\n".join(str(p) for p in adapter_candidates)
    )

print("Previous good adapter:", PREVIOUS_ADAPTER_DIR)


## 6. Load SmolVLM-256M and make the existing LoRA adapter trainable


In [ ]:
import gc
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import PeftModel

MODEL_CACHE_CANDIDATES = [
    PROJECT_ROOT / "model_cache",
    HOME / "EdgeVLM_finetune" / "model_cache",
]
MODEL_CACHE = next((p for p in MODEL_CACHE_CANDIDATES if p.exists()), MODEL_CACHE_CANDIDATES[0])
MODEL_CACHE.mkdir(parents=True, exist_ok=True)

processor = AutoProcessor.from_pretrained(
    str(PREVIOUS_ADAPTER_DIR),
    local_files_only=True,
)

load_options = {
    "cache_dir": str(MODEL_CACHE),
    "torch_dtype": torch.float16,
    "low_cpu_mem_usage": True,
}

try:
    base_model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        local_files_only=True,
        **load_options,
    )
    print("Base model loaded from the existing cache.")
except OSError:
    print("The cache is incomplete. Downloading only the missing base-model files...")
    base_model = AutoModelForVision2Seq.from_pretrained(MODEL_ID, **load_options)

model = PeftModel.from_pretrained(
    base_model,
    str(PREVIOUS_ADAPTER_DIR),
    is_trainable=True,
)
model = model.to("cuda")
model.config.use_cache = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print("Total parameters:", total)
print("Trainable LoRA parameters:", trainable)
print("Trainable percentage:", round(100 * trainable / total, 4), "%")


## 7. Create the Phase 2 image datasets


In [ ]:
from PIL import Image
from torch.utils.data import Dataset

INSTRUCTION = (
    "Classify the driver's visible state. "
    "Return exactly one label from: NORMAL, DROWSY, LOOKING_AWAY, or PHONE_USAGE. "
    "Do not provide an explanation or punctuation."
)


class DriverStateDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = Path(row["resolved_path"])
        with Image.open(image_path) as image_file:
            image = image_file.convert("RGB").copy()

        messages = [{
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": INSTRUCTION},
            ],
        }]
        return {
            "image": image,
            "messages": messages,
            "target": row["label"],
            "path": str(image_path),
        }


train_dataset = DriverStateDataset(train_df)
val_dataset = DriverStateDataset(val_df)
test_dataset = DriverStateDataset(test_df)

example = train_dataset[0]
print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Test examples:", len(test_dataset))
print("Example image:", example["path"])
print("Example target (used only as training output):", example["target"])


## 8. Build the multimodal training batch


In [ ]:
class SmolVLMDataCollator:
    def __init__(self, processor, image_size=512):
        self.processor = processor
        self.image_size = image_size

    def __call__(self, features):
        images = [[item["image"]] for item in features]
        full_texts = []
        prompt_texts = []

        for item in features:
            user_messages = item["messages"]
            full_messages = user_messages + [{
                "role": "assistant",
                "content": [{"type": "text", "text": item["target"]}],
            }]
            prompt_texts.append(
                self.processor.apply_chat_template(
                    user_messages,
                    add_generation_prompt=True,
                    tokenize=False,
                )
            )
            full_texts.append(
                self.processor.apply_chat_template(
                    full_messages,
                    add_generation_prompt=False,
                    tokenize=False,
                )
            )

        kwargs = {
            "return_tensors": "pt",
            "padding": True,
            "truncation": True,
            "size": {"longest_edge": self.image_size},
            "max_image_size": {"longest_edge": self.image_size},
        }
        batch = self.processor(text=full_texts, images=images, **kwargs)
        prompt_batch = self.processor(text=prompt_texts, images=images, **kwargs)

        labels = batch["input_ids"].clone()
        prompt_lengths = prompt_batch["attention_mask"].sum(dim=1)
        for row_index, prompt_length in enumerate(prompt_lengths):
            labels[row_index, : int(prompt_length)] = -100

        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100
        batch["labels"] = labels
        return batch


collator = SmolVLMDataCollator(processor, IMAGE_LONGEST_EDGE)

# This verifies forward and backward passes without updating the model weights.
sample_batch = collator([train_dataset[0]])
sample_batch = {
    key: value.to("cuda") if torch.is_tensor(value) else value
    for key, value in sample_batch.items()
}
model.train()
smoke_output = model(**sample_batch)
smoke_output.loss.backward()
model.zero_grad(set_to_none=True)
print("Non-updating smoke-test loss:", float(smoke_output.loss.detach().cpu()))
del sample_batch, smoke_output
gc.collect()
torch.cuda.empty_cache()


## 9. Continue fine-tuning on Phase 2

This stage intentionally uses the normal token loss. The starting adapter
already contains the earlier class-weighted learning, while the new Phase 2
training split contains substantially more DROWSY examples. Applying strong
inverse-frequency weights again could increase false alarms, especially for
LOOKING_AWAY.


In [ ]:
import time
from datetime import datetime, timedelta
from transformers import TrainingArguments, Trainer, TrainerCallback, set_seed
from transformers.trainer_utils import get_last_checkpoint

set_seed(SEED)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

PROGRESS_FILE = OUTPUT_ROOT / "training_progress.txt"
PROGRESS_HISTORY_FILE = OUTPUT_ROOT / "training_progress_history.log"


def format_duration(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


class ETAProgressCallback(TrainerCallback):
    """Write a terminal-friendly progress indicator with an estimated ETA."""

    def __init__(self, progress_file, history_file, update_every=25):
        self.progress_file = Path(progress_file)
        self.history_file = Path(history_file)
        self.update_every = int(update_every)
        self.started_at = None
        self.start_step = 0

    def on_train_begin(self, args, state, control, **kwargs):
        self.started_at = time.time()
        self.start_step = int(state.global_step)
        message = (
            f"Training started | step {state.global_step}/{state.max_steps} | "
            f"time {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
        )
        self.progress_file.write_text(message + "\n", encoding="utf-8")
        with self.history_file.open("a", encoding="utf-8") as handle:
            handle.write(message + "\n")
        print(message, flush=True)

    def on_step_end(self, args, state, control, **kwargs):
        step = int(state.global_step)
        if step != int(state.max_steps) and step % self.update_every != 0:
            return

        elapsed = time.time() - self.started_at
        completed_this_run = max(1, step - self.start_step)
        seconds_per_step = elapsed / completed_this_run
        remaining_steps = max(0, int(state.max_steps) - step)
        remaining_seconds = remaining_steps * seconds_per_step
        finish_time = datetime.now() + timedelta(seconds=remaining_seconds)
        percentage = 100.0 * step / max(1, int(state.max_steps))

        message = (
            f"{percentage:6.2f}% | step {step}/{state.max_steps} | "
            f"elapsed {format_duration(elapsed)} | "
            f"remaining {format_duration(remaining_seconds)} | "
            f"ETA {finish_time.strftime('%Y-%m-%d %H:%M:%S')}"
        )
        self.progress_file.write_text(message + "\n", encoding="utf-8")
        with self.history_file.open("a", encoding="utf-8") as handle:
            handle.write(message + "\n")
        print(message, flush=True)

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.started_at
        message = (
            f"100.00% | training finished at step {state.global_step} | "
            f"elapsed this run {format_duration(elapsed)} | "
            f"finished {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
        )
        self.progress_file.write_text(message + "\n", encoding="utf-8")
        with self.history_file.open("a", encoding="utf-8") as handle:
            handle.write(message + "\n")
        print(message, flush=True)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=25,
    disable_tqdm=True,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    optim="adamw_torch",
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    callbacks=[
        ETAProgressCallback(
            PROGRESS_FILE,
            PROGRESS_HISTORY_FILE,
            update_every=25,
        )
    ],
)

last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))
print("Resume checkpoint:", last_checkpoint or "None — starting Phase 2 training")
print("Starting Phase 2 fine-tuning...")
print("Live progress file:", PROGRESS_FILE)

train_result = trainer.train(resume_from_checkpoint=last_checkpoint)

print("Phase 2 fine-tuning completed.")
print(train_result)
trainer.save_model(str(FINAL_ADAPTER_DIR))
processor.save_pretrained(str(FINAL_ADAPTER_DIR))
print("Final Phase 2 adapter saved to:", FINAL_ADAPTER_DIR)


## 10. Reload the saved Phase 2 adapter for a clean test


In [ ]:
del trainer, model, base_model
gc.collect()
torch.cuda.empty_cache()

processor = AutoProcessor.from_pretrained(str(FINAL_ADAPTER_DIR), local_files_only=True)
base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    cache_dir=str(MODEL_CACHE),
    local_files_only=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
).to("cuda")
model = PeftModel.from_pretrained(base_model, str(FINAL_ADAPTER_DIR)).to("cuda")
model.eval()
model.config.use_cache = True

print("Reloaded Phase 2 adapter:", FINAL_ADAPTER_DIR)


## 11. Evaluate all Phase 2 test images

The result file is saved every 250 images. If the kernel stops, rerun the
notebook and this cell will continue from the saved row indices instead of
starting the test from zero.


In [ ]:
import time
import numpy as np
from tqdm.auto import tqdm

PREDICTIONS_CSV = OUTPUT_ROOT / "phase2_test_predictions.csv"
SAVE_EVERY = 250


def parse_prediction(raw_text):
    normalized = str(raw_text).strip().upper()
    exact_valid = normalized in LABELS
    cleaned = normalized.strip(" .,;:!?'\"")
    return (cleaned if cleaned in LABELS else "INVALID"), exact_valid


def predict_image(image_path):
    with Image.open(image_path) as image_file:
        image = image_file.convert("RGB").copy()

    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": INSTRUCTION},
        ],
    }]
    prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )
    inputs = processor(
        text=prompt,
        images=[image],
        return_tensors="pt",
        size={"longest_edge": IMAGE_LONGEST_EDGE},
        max_image_size={"longest_edge": IMAGE_LONGEST_EDGE},
    )
    inputs = {k: v.to("cuda") if torch.is_tensor(v) else v for k, v in inputs.items()}

    start = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=12,
            do_sample=False,
        )
    latency_ms = (time.perf_counter() - start) * 1000

    input_length = inputs["input_ids"].shape[-1]
    raw_output = processor.batch_decode(
        generated[:, input_length:],
        skip_special_tokens=True,
    )[0].strip()
    prediction, exact_valid = parse_prediction(raw_output)
    return raw_output, prediction, exact_valid, latency_ms


if PREDICTIONS_CSV.is_file():
    records = pd.read_csv(PREDICTIONS_CSV).to_dict("records")
    completed_indices = {int(row["row_index"]) for row in records}
    print("Resuming evaluation after", len(completed_indices), "saved rows.")
else:
    records = []
    completed_indices = set()

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

remaining = [index for index in range(len(test_df)) if index not in completed_indices]
for completed, index in enumerate(
    tqdm(remaining, desc="Evaluating all Phase 2 test images", mininterval=2.0),
    start=1,
):
    row = test_df.iloc[index]
    raw_output, prediction, exact_valid, latency_ms = predict_image(row["resolved_path"])
    records.append({
        "row_index": index,
        "frame_path": row["frame_path"],
        "resolved_path": row["resolved_path"],
        "true_label": row["label"],
        "raw_output": raw_output,
        "prediction": prediction,
        "exact_label_valid": exact_valid,
        "latency_ms": latency_ms,
    })

    if completed % SAVE_EVERY == 0:
        pd.DataFrame(records).sort_values("row_index").to_csv(PREDICTIONS_CSV, index=False)

predictions_df = pd.DataFrame(records).sort_values("row_index").drop_duplicates(
    subset=["row_index"], keep="last"
)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

print("Evaluation rows saved:", len(predictions_df))
print("Predictions file:", PREDICTIONS_CSV)


## 12. Print the final Phase 2 metrics


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support

if len(predictions_df) != len(test_df):
    raise RuntimeError(
        f"Evaluation is incomplete: {len(predictions_df)}/{len(test_df)} rows"
    )

y_true = predictions_df["true_label"].astype(str).to_numpy()
y_pred = predictions_df["prediction"].astype(str).to_numpy()
latencies = predictions_df["latency_ms"].astype(float).to_numpy()

precision, recall, f1, support = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=LABELS,
    zero_division=0,
)

unsafe_true = np.isin(y_true, ["DROWSY", "LOOKING_AWAY", "PHONE_USAGE"])
normal_true = y_true == "NORMAL"
unsafe_pred = np.isin(y_pred, ["DROWSY", "LOOKING_AWAY", "PHONE_USAGE"])

unsafe_to_safe_miss_rate = np.mean(np.isin(y_pred[unsafe_true], ["NORMAL", "INVALID"]))
false_alarm_rate = np.mean(unsafe_pred[normal_true])
exact_validity = predictions_df["exact_label_valid"].astype(bool).mean()
parser_failure = np.mean(y_pred == "INVALID")

matrix_labels = LABELS + ["INVALID"]
cm = confusion_matrix(y_true, y_pred, labels=matrix_labels)

print("\n========== PHASE 2 FINAL TEST RESULTS ==========")
print("Evaluation images:", len(predictions_df))
print("Exact-label validity rate:", round(exact_validity * 100, 2), "%")
print("Parser failure rate:", round(parser_failure * 100, 2), "%")
print("Overall accuracy:", round(accuracy_score(y_true, y_pred) * 100, 2), "%")
print("Balanced accuracy:", round(recall.mean() * 100, 2), "%")
print("Macro-F1:", round(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0) * 100, 2), "%")

print("\nPer-class results:")
for label, p, r, score, count in zip(LABELS, precision, recall, f1, support):
    print(f"{label}: precision={p:.3f}, recall={r:.3f}, F1={score:.3f}, samples={count}")

print("\nConfusion matrix:")
print(pd.DataFrame(cm, index=matrix_labels, columns=matrix_labels))
print("Unsafe-to-safe miss rate:", round(unsafe_to_safe_miss_rate * 100, 2), "%")
print("False-alarm rate:", round(false_alarm_rate * 100, 2), "%")
print("Mean latency (ms):", round(float(np.mean(latencies)), 2))
print("P95 latency (ms):", round(float(np.percentile(latencies, 95)), 2))
if torch.cuda.is_available():
    print("Peak GPU memory (MB):", round(torch.cuda.max_memory_allocated() / 1024**2, 2))


## 13. Optional: create a merged FP32 deployment folder

Keep this disabled until the Phase 2 results are accepted. Enabling it merges
the new adapter into the base model and creates the complete Hugging Face folder
that Students A and B can convert once on the Jetson.


In [ ]:
CREATE_MERGED_FP32 = False

if CREATE_MERGED_FP32:
    MERGED_DIR = OUTPUT_ROOT / "merged_model_256m_phase2_fp32"
    merged_model = model.merge_and_unload().float()
    merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
    processor.save_pretrained(str(MERGED_DIR))
    print("Merged FP32 model saved to:", MERGED_DIR)
else:
    print("Merge skipped. Set CREATE_MERGED_FP32=True only after accepting the results.")
